# 08 — Model Explainability (SHAP)

SHAP (SHapley Additive exPlanations) helps us understand WHY the model
makes specific predictions.

**Why this matters:**
- Feature importance tells us which features matter OVERALL
- SHAP tells us how each feature pushes a SPECIFIC prediction
- This is critical for trust — an operations manager needs to understand
  why a particular order was flagged as high risk

**Important note:** SHAP values show feature CONTRIBUTION to the prediction,
not causation. Saying "distance contributed to the high risk prediction"
is not the same as saying "distance caused the delay."

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
import sys
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

plt.style.use('seaborn-v0_8-whitegrid')

## 8.1 Load Model and Prepare Data

In [ ]:
model = joblib.load("../models/best_model.pkl")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

# Drop same high-cardinality columns as in training
high_card_cols = ["Order City", "Order State", "Customer City", "Customer State"]
X_test = X_test.drop(columns=[c for c in high_card_cols if c in X_test.columns])

# Get the preprocessed features (after the pipeline's transformer)
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

X_test_transformed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print(f"Transformed features: {X_test_transformed.shape[1]}")
print(f"Model type: {type(classifier).__name__}")


## 8.2 Tree-Based Feature Importance (Built-in)

First, let's look at the model's built-in feature importance.
This is based on how much each feature reduces impurity during training.

In [ ]:
importances = classifier.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

print("Top 20 Features (tree-based importance):")
print(feature_importance.head(20).to_string(index=False))

# Plot
plt.figure(figsize=(10, 8))
top20 = feature_importance.head(20)
sns.barplot(data=top20, x="Importance", y="Feature", palette="viridis")
plt.title("Top 20 Features — Tree-Based Importance")
plt.tight_layout()
plt.show()

## 8.3 SHAP Analysis

SHAP provides a more nuanced view than tree-based importance.
It shows the DIRECTION and MAGNITUDE of each feature's effect.

We use `TreeExplainer` which is optimized for tree-based models like ExtraTrees.

In [ ]:
# Use a representative sample for SHAP
np.random.seed(42)
sample_idx = np.random.choice(len(X_test_transformed), size=min(25, len(X_test_transformed)), replace=False)
X_sample = X_test_transformed[sample_idx]

print(f"Computing SHAP values for {len(X_sample)} samples...")
explainer = shap.TreeExplainer(classifier)
shap_values = explainer.shap_values(X_sample)

print("SHAP values computed successfully.")


## 8.4 SHAP Summary Plot — Global Feature Importance

This shows which features have the biggest impact on predictions OVERALL.
Each dot represents one sample. Red = high feature value, Blue = low.

In [ ]:
# For binary classification, shap_values may be a list [class_0, class_1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # Use class 1 (late delivery)
else:
    shap_vals = shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals,
    X_sample,
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title("SHAP Summary Plot — Feature Impact on Late Delivery Prediction")
plt.tight_layout()
plt.show()

## 8.5 SHAP Bar Plot — Mean Absolute Impact

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals,
    X_sample,
    feature_names=feature_names,
    plot_type="bar",
    max_display=20,
    show=False
)
plt.title("SHAP Mean Absolute Impact")
plt.tight_layout()
plt.show()

## 8.6 Individual Prediction Explanations

Let's explain specific predictions to understand the model's reasoning
for individual orders.

In [ ]:
# Find a high-risk prediction and a low-risk prediction
y_prob_sample = model.predict_proba(X_test.iloc[sample_idx])[:, 1]

high_risk_idx = np.argmax(y_prob_sample)
low_risk_idx = np.argmin(y_prob_sample)

print(f"High-risk example: predicted probability = {y_prob_sample[high_risk_idx]:.3f}")
print(f"Low-risk example: predicted probability = {y_prob_sample[low_risk_idx]:.3f}")

In [ ]:
# Explain the high-risk prediction
print("\n" + "=" * 60)
print("HIGH-RISK ORDER EXPLANATION")
print("=" * 60)
print(f"Predicted late delivery probability: {y_prob_sample[high_risk_idx]:.3f}")
print(f"Actual outcome: {'Late' if y_test.iloc[sample_idx[high_risk_idx]] == 1 else 'On Time'}")
print("\nTop contributing factors:")

# Get SHAP values for this prediction
high_risk_shap = shap_vals[high_risk_idx]
top_factors = pd.DataFrame({
    "Feature": feature_names,
    "SHAP Value": high_risk_shap
}).sort_values("SHAP Value", key=abs, ascending=False).head(10)

for _, row in top_factors.iterrows():
    direction = "↑ increases" if row["SHAP Value"] > 0 else "↓ decreases"
    print(f"  {row['Feature']:40s} {direction} risk (SHAP={row['SHAP Value']:+.4f})")

In [ ]:
# Waterfall plot for high-risk prediction
shap_explanation = shap.Explanation(
    values=shap_vals[high_risk_idx],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
    feature_names=feature_names
)

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_explanation, max_display=12, show=False)
plt.title("SHAP Waterfall — High Risk Order")
plt.tight_layout()
plt.show()

## 8.7 Comparing SHAP vs Tree-Based Importance

In [ ]:
# Compare top features from both methods
tree_top = feature_importance.head(10)["Feature"].tolist()

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Abs_SHAP": np.abs(shap_vals).mean(axis=0)
}).sort_values("Mean_Abs_SHAP", ascending=False)

shap_top = shap_importance.head(10)["Feature"].tolist()

print("Top 10 Features Comparison:")
print("=" * 50)
print(f"{'Tree-Based':30s} | {'SHAP':30s}")
print("-" * 63)
for i in range(10):
    print(f"{tree_top[i]:30s} | {shap_top[i]:30s}")

overlap = set(tree_top) & set(shap_top)
print(f"\nOverlap: {len(overlap)} of 10 features appear in both top-10 lists")

## Summary

**Key takeaways:**

1. **Shipping Mode** is consistently the most important feature across both
   tree-based importance and SHAP analysis

2. SHAP reveals the DIRECTION of impact — for example, Standard Class
   shipping pushes predictions toward higher late delivery risk

3. Individual predictions can be explained: for any order, we can show
   which factors contributed most to its risk score

4. The agreement between tree-based and SHAP importance increases our
   confidence in the feature rankings

**Limitation:** SHAP shows CONTRIBUTION to prediction, not causation.
The model learned associations from historical data — it doesn't know
why certain factors lead to delays.

**Next step:** Demand forecasting